# Practical 8: Retrieval Evaluation Metrics

**Aim:** Rank documents for q1 and calculate Precision, Recall, F1, E-measure, and NDCG from relevance judgments.

This notebook reads files from `data/`. The included documents are a small demonstration set. For the real-world lab, use the Cranfield documents, queries, and relevance assessments from the [University of Glasgow collection page](https://ir.dcs.gla.ac.uk/resources/test_collections/cran/).

Install scikit-learn if needed: `python -m pip install scikit-learn`. Run cells from top to bottom with this practical folder as the working directory.


## 1. Read q1, documents, and relevance judgments

Each qrels line has a query ID, document ID, and relevance grade. The code keeps q1 judgments.


In [ ]:
from pathlib import Path
import math

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = Path("data")
TOP_K = 3
documents = {file.stem: file.read_text(encoding="utf-8")
             for file in sorted((DATA / "documents").glob("*.txt"))}
query_line = (DATA / "query.txt").read_text(encoding="utf-8").strip()
query = query_line.split(maxsplit=1)[1]
qrels = {}
for line in (DATA / "qrels.txt").read_text(encoding="utf-8").splitlines():
    query_id, doc_id, grade = line.split()
    if query_id == "1":
        qrels[doc_id] = max(0, int(grade))
print("Query q1:", query)
print("Documents:", len(documents), "Relevant judgments:", len(qrels))


## 2. Rank documents for q1

TF-IDF and cosine similarity make the ranked list. Relevance labels are used afterwards to evaluate the ranking.


In [ ]:
doc_ids = list(documents)
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(documents.values())
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, doc_vectors).ravel()
ranked = sorted(zip(doc_ids, scores), key=lambda item: item[1], reverse=True)
top_results = ranked[:TOP_K]
for rank, (doc_id, score) in enumerate(top_results, start=1):
    print(rank, doc_id, round(score, 3), "relevance:", qrels.get(doc_id, 0))


## 3. Calculate the measures

Precision and recall use binary relevance (`grade > 0`). NDCG uses graded relevance, discounted by result position.


In [ ]:
relevant_found = sum(qrels.get(doc_id, 0) > 0 for doc_id, score in top_results)
total_relevant = sum(grade > 0 for grade in qrels.values())
precision = relevant_found / TOP_K
recall = relevant_found / total_relevant
f_measure = 2 * precision * recall / (precision + recall) if precision + recall else 0
e_measure = 1 - f_measure

def dcg(grades):
    return sum((2**grade - 1) / math.log2(rank + 1)
               for rank, grade in enumerate(grades, start=1))

retrieved_grades = [qrels.get(doc_id, 0) for doc_id, score in top_results]
ideal_grades = sorted(qrels.values(), reverse=True)[:TOP_K]
ndcg = dcg(retrieved_grades) / dcg(ideal_grades) if dcg(ideal_grades) else 0

print("Precision@3:", round(precision, 3))
print("Recall@3:   ", round(recall, 3))
print("F-measure:  ", round(f_measure, 3))
print("E-measure:  ", round(e_measure, 3))
print("NDCG@3:     ", round(ndcg, 3))


## A/B testing and feedback

A/B testing randomly compares the current search interface (A) with a changed version (B). Give both groups similar tasks and compare task success, relevant results found, time to find an item, and user feedback. Click rate alone does not prove search quality improved.

Example feedback themes for class discussion:

- Amazon-style product search: “The size filter is selected, but I still see products in other sizes.”
- Netflix-style recommendations: “I keep seeing titles I have already watched.”

These are illustrative examples, not direct user quotes. A real test collects participant feedback and compares it with retrieval measures.


## Viva questions with answers

1. **What is precision?** The fraction of retrieved documents that are relevant.
2. **What is recall?** The fraction of all relevant documents that were retrieved.
3. **What is F-measure?** A combined score based on precision and recall; this practical uses F1.
4. **What is E-measure here?** The complement of F1, calculated as `1 - F1`.
5. **What is NDCG?** A measure that rewards relevant results near the top and compares them with an ideal ranking.
6. **What is a qrel?** A relevance judgment for a query-document pair.
7. **Why use TF-IDF?** It turns documents and the query into vectors for similarity ranking.
8. **What does cosine similarity do?** It scores how similar the query vector is to each document vector.
9. **What is A/B testing?** A controlled comparison of two interface versions using different user groups.
10. **Why collect feedback as well as scores?** Scores measure retrieval performance; feedback helps explain user experience problems.
